In [2]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    temperature=0, 
    #groq_api_key='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', 
    model_name="llama-3.3-70b-versatile"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)


The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [14]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/senior-additional-demand-equipment-rep/job/R-64409")
page_data = loader.load().pop().page_content
print(page_data)






















Senior Additional Demand & Equipment Rep










































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Pr

In [16]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [18]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Senior Additional Demand & Equipment Rep',
 'experience': ['Experience in retail buying, merchandising, sales business planning, or sales business analysis working with forecast and inventory during a couple of seasons',
  'Project management experience working cross functionally',
  'Experience in successfully implementing various sales plans, procedures, and methodologies and experience in utilizing financial decision making, models and translating results into business plans and accurate revenue forecasts'],
 'skills': ['Strong knowledge of the athletic industry both from product knowledge and marketplace trends',
  'Exceptional verbal and written communication skills, with ability to clearly articulate goals and objectives',
  'Ability to prepare, plan and deliver clear and persuasive presentations, and think strategically by providing problem-solving solutions',
  'Ability to use financial data to make decisions and maximize account profitability',
  'Ability to develop 

In [20]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [22]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


In [28]:
job = json_res
links = collection.query(query_texts=job['skills'], n_results=2).get('metadatas', [])
links

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/ios-ar-portfolio'}],
 [{'links': 'https://example.com/ios-ar-portfolio'},
  {'links': 'https://example.com/devops-portfolio'}],
 [{'links': 'https://example.com/python-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/kotlin-backend-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}]]

In [30]:
job

{'role': 'Senior Additional Demand & Equipment Rep',
 'experience': ['Experience in retail buying, merchandising, sales business planning, or sales business analysis working with forecast and inventory during a couple of seasons',
  'Project management experience working cross functionally',
  'Experience in successfully implementing various sales plans, procedures, and methodologies and experience in utilizing financial decision making, models and translating results into business plans and accurate revenue forecasts'],
 'skills': ['Strong knowledge of the athletic industry both from product knowledge and marketplace trends',
  'Exceptional verbal and written communication skills, with ability to clearly articulate goals and objectives',
  'Ability to prepare, plan and deliver clear and persuasive presentations, and think strategically by providing problem-solving solutions',
  'Ability to use financial data to make decisions and maximize account profitability',
  'Ability to develop 

In [32]:
job['skills']

['Strong knowledge of the athletic industry both from product knowledge and marketplace trends',
 'Exceptional verbal and written communication skills, with ability to clearly articulate goals and objectives',
 'Ability to prepare, plan and deliver clear and persuasive presentations, and think strategically by providing problem-solving solutions',
 'Ability to use financial data to make decisions and maximize account profitability',
 'Ability to develop seasonal assortments, forecasts, and scenario plans']

In [34]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Enhancing Nike's Retail Operations with AtliQ's Expertise

Dear Hiring Manager,

I came across the job description for a Senior Additional Demand & Equipment Rep at Nike, and I was impressed by the role's focus on driving business growth through data-driven decision making and strategic planning. As a business development executive at AtliQ, I believe our company's capabilities align perfectly with Nike's goals.

At AtliQ, we specialize in developing tailored solutions that foster scalability, process optimization, and cost reduction. Our expertise in AI and software consulting can help Nike enhance its retail operations, maximize in-season business, and expand equipment sales. Our team can assist in developing seasonal assortments, forecasts, and scenario plans, as well as provide data-driven insights to inform business decisions.

I'd like to highlight a few examples from our portfolio that demonstrate our relevance to Nike's needs:

* Our machine learning expertise, showcas